In [ ]:
pip install opacus

In [ ]:
pip install transformers


In [3]:
pip install accelerate>=0.26.0

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install torch

In [1]:
import torch
import csv
import os
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.optim import SGD, AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from opacus import PrivacyEngine  # Assuming you're using Opacus for DP
import collections
device = "cuda"
MAX_LEN = 768

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token
class CSVMIMIC(Dataset):
    def __init__(self, dataset_dir="NOTEEVENTS_deanonymized.csv", notes_amount=-1, max_length=128):
        self.notes = []

        with open(dataset_dir, newline="", encoding="ISO-8859-1") as csvfile:
            note_row = csv.reader(csvfile)
            next(note_row)  # Skip header
            for row in tqdm(note_row):
                text = row[1]
                text_without_first_words= " ".join(text.split()[2:])
                encoded = tokenizer.encode(text_without_first_words, max_length=max_length, truncation=True)
                self.notes.append(torch.tensor(encoded, dtype=torch.long))
                if len(self.notes) == notes_amount:
                    break

    def __len__(self):
        return len(self.notes)

    def __getitem__(self, idx):
        return self.notes[idx]


def collate_fn(batch, pad_token_id):
    input_ids = pad_sequence(
        batch,
        batch_first=True,
        padding_value=pad_token_id
    )

    attention_mask = (input_ids != pad_token_id).long()
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask
    }






In [2]:
from enum import Enum, auto

class TrainingMode(Enum):
    STANDARD = auto()     
    DIAGNOSTIC = auto()   # DP Opacus framework but clipping only, noise=0
    DP = auto()           # DP Opacus


def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    steps = 0

    with torch.no_grad():
        for entry in tqdm(dataloader, total=len(dataloader)):
            input_ids = entry["input_ids"].to(device)
            attention_mask = entry["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=input_ids
            )

            loss = outputs.loss  # already mask-aware + averaged
            total_loss += loss.item()
            steps += 1

    avg_loss = total_loss / steps if steps > 0 else float("nan")
    perplexity = torch.exp(torch.tensor(avg_loss))

    return avg_loss, perplexity.item()





    
def train(
    dataset,
    model,
    batch_size=8,
    epochs=20,
    lr=1e-5,
    warmup_steps=2500,
    device="cuda",
    output_dir=".",
    output_prefix="model",
    save_model_on_epoch=False,
    mode=TrainingMode.STANDARD,
    gc=1,
    epsilon=10,
    vocab_size = None,
    checkpoint = None
):
    model = model.to(device)
    model.train()

    optimizer = SGD(model.parameters(), lr=lr)

    # Train/val split
    val_ratio = 0.1
    train_size = int(len(dataset) * (1 - val_ratio))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        collate_fn=lambda x: collate_fn(x, tokenizer.pad_token_id)
    )
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda x: collate_fn(x, tokenizer.pad_token_id)
    )
    
    criterion = torch.nn.CrossEntropyLoss()
    if mode == TrainingMode.DP:
        privacy_engine = PrivacyEngine(accountant="rdp")
        print("Delta is inverse of length of dataloader:", len(train_dataloader))
        model, optimizer, _, train_dataloader = privacy_engine.make_private_with_epsilon(
            module=model,
            optimizer=optimizer,
            data_loader=train_dataloader,
            max_grad_norm=gc,
            target_epsilon=epsilon,
            target_delta = 1/(len(train_dataloader)*10),
            epochs=epochs,
            criterion=criterion,
            grad_sample_mode="ghost",
            poisson_sampling=True,      
        )
    elif mode == TrainingMode.DIAGNOSTIC:
        privacy_engine = PrivacyEngine(accountant="rdp")
        model, optimizer, dataloader = privacy_engine.make_private(
                    module=model,
                    optimizer=optimizer,
                    data_loader=train_dataloader,
                    noise_multiplier=0,
                    max_grad_norm=gc
                 )

        
    model.to(device)
    if checkpoint:
        model.load_state_dict(checkpoint)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0.1 * len(train_dataloader),
        num_training_steps=len(train_dataloader)* epochs
    )

    metrics = {"train_loss": [], "val_loss": [], "val_ppl": []}
    for epoch in range(epochs):
        print(f"\nTraining epoch {epoch+1}/{epochs}")
        model.train()
        
        epoch_loss = 0
        steps = 0
        
        for entry in tqdm(train_dataloader, total=len(train_dataloader)):
            if isinstance(entry, list):
                print("WARNING: Skipping list entry")
                continue
            
            input_ids = entry["input_ids"].to(device)
            attention_mask = entry["attention_mask"].to(device)
            

            optimizer.zero_grad()
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=input_ids 
            )
            loss = outputs.loss
            
    
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"WARNING: Invalid loss detected: {loss.item()}, skipping batch")
                continue
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            steps += 1
        
        # Step scheduler ONCE per epoch (after all batches)
        scheduler.step()
        
        avg_train_loss = epoch_loss / steps if steps > 0 else float("nan")
        val_loss, val_ppl = evaluate(model, val_dataloader, device)
        
        print(
            f"Train Loss: {avg_train_loss:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Val PPL: {val_ppl:.2f}"
        )
        
        metrics["train_loss"].append(avg_train_loss)
        metrics["val_loss"].append(val_loss)
        metrics["val_ppl"].append(val_ppl)
        
        if mode == TrainingMode.DP:
            delta = 1e-5
            calculated_epsilon = privacy_engine.get_epsilon(delta)
            print(f"Privacy budget: ε={epsilon:.2f}, δ={delta}")
            
            if calculated_epsilon > epsilon:
                print(f"Privacy budget exceeded (ε={calculated_epsilon:.2f} > {epsilon}). Stopping.")
                break
        
        if save_model_on_epoch:
            torch.save(
                model.state_dict(),
                os.path.join(output_dir, f"{output_prefix}-epoch{epoch+20}.pt"),
            )


    return model, metrics

In [4]:
dataset = CSVMIMIC(dataset_dir="SUBJECT_ID_to_NOTES_1b.csv", notes_amount=100000)

99999it [01:07, 1476.93it/s]


In [ ]:
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
model, metric = train(dataset, model, batch_size=16, epochs=25, lr=0.01)

In [ ]:
save_dir = "checkpoints/finalSGD_GPT2"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)


In [5]:
import torch
import numpy as np
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import csv
from tqdm import tqdm
from itertools import islice
import json
import re
import numpy as np
import torch
import torch.nn.functional as F
import numpy as np
import math
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import defaultdict
import matplotlib.pyplot as plt

P_PARAMETER = 0.9 #n-p discoverable extraction p parameter.

def compute_np_discoverable_extraction(secret_prob, p=P_PARAMETER):
    num = np.log2(1-p)
    den = np.log2(1-secret_prob)
    return num/den
    


def get_probability(logprob: float) -> float:
    return round(math.exp(logprob), 2)
    

def sorting_ranking(secrets_probabilities, secrets_paired_to_context, tokenizer, allow_filler_tokens=False):
    standardized = {tuple(k): v for k, v in secrets_probabilities.items()}
    ranking = sorted(standardized.items(), key=lambda x: x[1], reverse=True)  #Highest probability first
    correct_secrets = secrets_paired_to_context
    enriched_ranking = []
    
    for key, probability in ranking:
        phrase = ''.join(key)
        match = False
        if allow_filler_tokens:
            correct_secret = find_matching_secret(phrase, correct_secrets)
            if correct_secret:
                match = True
                phrase = correct_secret
                
        else:
            match = (phrase in correct_secrets)
            
        attempts = compute_np_discoverable_extraction(probability)
        enriched_ranking.append({
            "decoded": phrase,
            "probability": probability,
            "attempts": attempts,
            "match": match
        })

    return enriched_ranking

def is_prefix_of_any(prefix, targets):
    prefix_len = prefix.size(0)
    return any(
        target.size(0) >= prefix_len
        and torch.equal(target[:prefix_len], prefix)
        for target in targets
    )

def compute_renyi_divergence_alpha_and_ranking(
    sequences,
    secrets_paired_to_context,
    alpha=0.5,
    dp=False,
    context="default context ",
    KL=False,
    checkpoint_dir="",
    device="cuda",
    topk=False,
    allow_filler_tokens=False,
    rank_among_R=True,
    k=1000
):
    """PyTorch version of computing Rényi divergence (OLMo-3)"""

    
    model_name="openai-community/gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if isinstance(checkpoint_dir, str):
        prediction_model = AutoModelForCausalLM.from_pretrained(
            checkpoint_dir,
            torch_dtype="auto"
        ).to(device)
    elif isinstance(checkpoint_dir, collections.OrderedDict):
        prediction_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto"
        ).to(device)
        prediction_model.load_state_dict(checkpoint_dir)

    prediction_model.eval()

    tokenized_context = tokenizer.encode(
        context,
        add_special_tokens=False
    )
    context_len = len(tokenized_context)

    if topk:
        secrets_prob, all_sequence_probs, sequence_count = calculate_topk_probabilities(tokenizer, prediction_model, tokenized_context, sequences, secrets_paired_to_context, allow_filler_tokens=allow_filler_tokens, rank_among_R=rank_among_R, k=k)
    else:
        secrets_prob, all_sequence_probs, sequence_count = calculate_greedy_probabilities(tokenizer, prediction_model, tokenized_context, sequences, secrets_paired_to_context)

    print(f"Total sequence count: {sequence_count}")

    ranking = sorting_ranking(
        secrets_prob,
        secrets_paired_to_context,
        tokenizer,
        allow_filler_tokens=allow_filler_tokens
    )

    if sequence_count == 0:
        print("No secrets found for the specified key.")
        return None, None, None

    divergence = calculate_divergence(all_sequence_probs, sequence_count, KL=KL, alpha=alpha)

    return divergence, ranking, all_sequence_probs



def calculate_divergence(all_sequence_probs, sequence_count, epsilon=1e-12, KL=True, alpha=0.5):
    total_prob = sum(all_sequence_probs)
    try:
        normalized_probs = [(p + epsilon) / total_prob for p in all_sequence_probs]
    except:
        print("WARNING: Total_prob is 0")
        normalized_probs = []

    uniform_prob = 1.0 / sequence_count

    if KL:
        divergence = sum(
            p * np.log2(p / uniform_prob)
            for p in normalized_probs
        )
    else:
        sum_product = sum(
            (p ** alpha) * (uniform_prob ** (1 - alpha))
            for p in normalized_probs
        )
        divergence = np.log2(sum_product) / (alpha - 1)
    return divergence
def calculate_topk_probabilities(tokenizer, prediction_model, tokenized_context, sequences_to_eval, secrets_paired_to_context, device="cuda", k=1000, depth=5, max_batch_size=128, rank_among_R=True, allow_filler_tokens=False):
    """
    rank_among_R: If true, we calculate the probabilities of the sequences to eval. If false, we calculate the probabilities of the generated sequences
    allow_filler_tokens: If false, we prune branches that are not substrings of sequences, if True, we do not prune since filler tokens are allowed.
    """
    all_sequence_probs = []
    target_token_seqs = [
                        torch.tensor(tokenizer.encode(s, add_special_tokens=False), device=device)
                        for s in sequences_to_eval]
    sequence_count = 0
    secrets_prob = {}
    prediction_model.eval()
    initial_input = torch.tensor([tokenized_context], device=device)
    words_probabilities = defaultdict(int)
    # each element: (token_ids, cumulative_logprob)
    sequences = [(initial_input[0], 0.0)]

    for step in range(depth):
        print("Current depth:", step)
        new_sequences = []

        # Process sequences in batches
        for batch_start in range(0, len(sequences), max_batch_size):
            batch_end = min(batch_start + max_batch_size, len(sequences))
            batch_sequences = sequences[batch_start:batch_end]

            # Stack batch sequences
            batch_seqs = torch.stack([seq for seq, _ in batch_sequences])  # [batch_size, seq_len]
            batch_logprobs = torch.tensor([logprob for _, logprob in batch_sequences], device=device)  # [batch_size]

            with torch.no_grad():
                logits = prediction_model(batch_seqs).logits[:, -1, :]  # [batch_size, vocab_size]
                log_probs = torch.log_softmax(logits, dim=-1)  # [batch_size, vocab_size]

            # Get top-k for each sequence in the batch

            topk_logprobs, topk_indices = torch.topk(log_probs, k, dim=-1)  # Both: [batch_size, k]

            # Expand sequences
            for i in range(len(batch_sequences)):
                seq = batch_seqs[i]
                seq_logprob = batch_logprobs[i].item()
                for j in range(k):
                    token_id = topk_indices[i, j].item()
                    token_logprob = topk_logprobs[i, j].item()
                    new_seq = torch.cat([seq, torch.tensor([token_id], device=device)])
                    new_seq_withoutcontext = new_seq[len(tokenized_context):]
                    has_prefix_match = is_prefix_of_any(
                                        new_seq_withoutcontext,
                                        target_token_seqs
                                        )
                    if allow_filler_tokens or (not rank_among_R) or has_prefix_match:
                        new_sequences.append((new_seq, seq_logprob + token_logprob + 1e-12))

        sequences = new_sequences

    secrets_probabilities = defaultdict(int)
    sequence_count = 0
    print(sequences[:50])
    sorted_sequences = sorted(sequences, key=lambda x: x[1], reverse=True)
    if rank_among_R:
        for seq, logprob in sorted_sequences:
            sequence_string = tokenizer.decode(seq, add_special_tokens=False)
            if not allow_filler_tokens:
                for eval_seq in sequences_to_eval:
                    if eval_seq in sequence_string:
                        words_probabilities[eval_seq] += logprob
            else:
                found_sequence = find_matching_secret(sequence_string, sequences_to_eval)
                if found_sequence:
                    print("Generated: ",sequence_string, "with logprob", logprob )
                    words_probabilities[found_sequence] += logprob

            
        for word in words_probabilities.keys(): #Converting logprobs to probabilities
           words_probabilities[word] = get_probability(words_probabilities[word])
            
        

    else: #Calculate probabilities of secrets among generated sequences
        for seq, logprob in sorted_sequences:
            sequence_string = tokenizer.decode(seq[len(tokenized_context):], add_special_tokens=False)
            words_probabilities[sequence_string] += logprob

            
        for words in words_probabilities.keys(): #Converting logprobs to probabilities
           words_probabilities[words] = get_probability(words_probabilities[words])

    sequence_count = len(words_probabilities)
    probs = list(words_probabilities.values())

    return words_probabilities, probs, sequence_count


def calculate_greedy_probabilities(tokenizer, prediction_model, tokenized_context, sequences, secrets_paired_to_context, device="cuda"):
    all_sequence_probs = []
    sequence_count = 0
    secrets_prob = {}
    context_len = len(tokenized_context)
    for secret in sequences:
        encoded_secret = tokenizer.encode(
            secret,
            add_special_tokens=False
        )

        sequence_count += 1

        max_length = prediction_model.config.max_position_embeddings
        max_secret_len = max_length - context_len

        full_input = tokenized_context + encoded_secret[:max_secret_len]

        tokens = torch.tensor(
            [full_input],
            device=device
        )

        with torch.no_grad():
            outputs = prediction_model(tokens)
            logits = outputs.logits  # (1, seq_len, vocab)

        logits = logits - logits.max(dim=-1, keepdim=True).values
        predictions_prob = F.softmax(logits, dim=-1)

        log_prob = 0.0
        for k in range(context_len, len(full_input)):
            token = full_input[k]
            if token == tokenizer.eos_token_id:
                break

            prob = predictions_prob[0, k - 1, token].item()
            log_prob += np.log2(prob)

        sequence_prob = 2 ** log_prob
        secrets_prob[secret] = sequence_prob
        all_sequence_probs.append(sequence_prob)

    return secrets_prob, all_sequence_probs, sequence_count

def calculate_exposure(secrets_ranking):
    space_size = len(secrets_ranking)
    avg_rank = calculate_avg_rank(secrets_ranking)
    print("The average rank of canary sequences is", avg_rank)
    return np.log2(space_size) - np.log2(avg_rank)

def calculate_avg_rank(secrets_ranking):
    """
     enriched_ranking.append({
            "decoded": phrase,
            "probability": probability,
            "match": match
        })"""
    true_match_ranks = []
    for pos in range(len(secrets_ranking)):
        if secrets_ranking[pos]["match"]==True:
            true_match_ranks.append(pos+1)

    avg_rank = np.sum(true_match_ranks) / len(true_match_ranks)
    return avg_rank



def contains_ordered_tokens(short, long): 
    short_tokens = short.split()
    long_tokens = long.split()

    it = iter(long_tokens)
    return all(token in it for token in short_tokens)


def find_matching_secret(phrase, correct_secrets):
    for secret in correct_secrets:
        if contains_ordered_tokens(secret, phrase):
            return secret
        
    return None


from matplotlib.ticker import ScalarFormatter
def plot_suffix_ranking(
    prefix,
    ranking,
    *,
    log_scale=True,
    max_items=None
):
    """
    Plot horizontal bar chart of suffix probabilities for a given prefix.

    Parameters
    ----------
    prefix : str
        The prefix used to generate the ranking.
    ranking : list[dict]
        List of entries with keys: decoded, probability, match.
    log_scale : bool
        Whether to use logarithmic x-axis.
    max_items : int or None
        Limit number of suffixes shown (top-k).
    """

    if max_items is not None:
        ranking = ranking[:max_items]

    words = [entry["decoded"].strip() for entry in ranking]
    probs = [entry["probability"] for entry in ranking]

    indices = np.arange(len(probs))
    colors = ["red" if entry.get("match", False) else "blue"
              for entry in ranking]

    plt.figure(figsize=(12, 0.4 * len(words)))

    plt.barh(indices, probs, color=colors)
    ax = plt.gca()
    if log_scale:
        plt.xscale("log")
    

    plt.ylabel("Suffix")
    plt.title(f"Suffix probabilities for prefix: '{prefix}'\n(red = canary)")

    plt.yticks(indices, words)
    plt.gca().invert_yaxis()  

    plt.tight_layout()
    plt.show()


samples_max_length = 5
context_len = 3 #len in tokens

ERROR! Session/line number was not unique in database. History logging moved to new session 101


In [5]:
prefixes_suffixes = [
    {"prefix":"Lubertha Dansie respiratory care: pt.Lubertha Dansie intubated in ew for airway protection. Lubertha Dansie transferred to",
     "suffix": "micu"},
    {"prefix": "Carmela Durso a: cont with anxiety and is successfully treated with",
     "suffix": "ativan"},
    {"prefix": "Kenton Lynn date of birth: sex: m service: medicine chief complaint: admitted from rehabilitation for hypotension (systolic blood pressure to the s) and decreased urine output.Kenton Lynn history of present illness: the patient is a -year-old male who had been hospitalized at the from through of after undergoing a left ", 
     "suffix":"femoral-at bypass"},
    {"prefix": "Lyndell Martire csru admission note mrs martire is a year old woman who presented to the ew today with increasing sob and chest pain.Lyndell Martire nkda neuro: sedated on",
     "suffix": "propofol"},
]



# Base GPT-2

In [ ]:

mimic_sequences = []
results = []
for prefix_suffix in tqdm(prefixes_suffixes):

    secrets_paired_to_context = prefix_suffix["suffix"]


    context = prefix_suffix["prefix"]

    divergence, ranking, _ = compute_renyi_divergence_alpha_and_ranking(
        sequences=[],
        secrets_paired_to_context = secrets_paired_to_context,
        alpha=1,
        context=context,
        KL=True,
        topk=True,
        k=50,
        checkpoint_dir="openai-community/gpt2",
        allow_filler_tokens=True,
        rank_among_R=True)
    print(f"Divergence with uniform distribution {divergence} using prefix {context}")
    if ranking:
        exposure = calculate_exposure(ranking)
        print("Exposure", exposure)
    else:
        exposure = np.log2(len(ranking))
    print(ranking[:10])
    plot_suffix_ranking(
        context,
        ranking,
        max_items=20  
    )
    results.append(ranking)

    

# Fine-tuned SGD GPT2

In [ ]:
save_dir = "checkpoints/finalSGD_GPT2"
mimic_sequences = []
results = []
for prefix_suffix in tqdm(prefixes_suffixes):

    secrets_paired_to_context = prefix_suffix["suffix"]

    context = prefix_suffix["prefix"]
    print(context, secrets_paired_to_context)
    divergence, ranking, _ = compute_renyi_divergence_alpha_and_ranking(
        sequences=[],
        secrets_paired_to_context = secrets_paired_to_context,
        alpha=1,
        context=context,
        KL=True,
        topk=True,
        k=10,
        checkpoint_dir=save_dir,
        allow_filler_tokens=True,
        rank_among_R=True)
    print(f"Divergence with uniform distribution {divergence} using prefix {context}")
    if ranking:
        exposure = calculate_exposure(ranking)
        print("Exposure", exposure)
        plot_suffix_ranking(
            context,
            ranking,
            max_items=20  
        )
        print(ranking[:10])

        results.append(ranking)
    else:
        exposure= np.log2(len(secrets_paired_to_context))
  

    

In [ ]:
results[0]

# DP-SGD

In [ ]:
from opacus.validators import ModuleValidator

model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
model.train()
ModuleValidator.validate(model, strict=True)


model = ModuleValidator.fix(model)


Comparing SGD model to DP-SGD with noise_multiplier=0 to identify gradient clipping

In [ ]:

model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
vocab_size = model.config.vocab_size

model.tie_weights = False
model.lm_head.weight = torch.nn.Parameter(
    model.transformer.wte.weight.clone()
)
model, metric = train(dataset, model, batch_size=1, epochs=25, lr=0.01, mode=TrainingMode.STANDARD)

Gradient clipping sweeps:

In [ ]:

model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
vocab_size = model.config.vocab_size

model.tie_weights = False
model.lm_head.weight = torch.nn.Parameter(
    model.transformer.wte.weight.clone()
)
model, metric = train(dataset, model, batch_size=1, epochs=25, lr=0.01, mode=TrainingMode.DIAGNOSTIC, gc=1, vocab_size=vocab_size)

Found gradient clipping parameter: gc=1

In [3]:
dataset = CSVMIMIC(dataset_dir="SUBJECT_ID_to_NOTES_1b.csv", notes_amount=100000)

99999it [01:08, 1466.30it/s]


In [ ]:

model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
vocab_size = model.config.vocab_size

model.tie_weights = False
model.lm_head.weight = torch.nn.Parameter(
    model.transformer.wte.weight.clone()
)
checkpoint = torch.load("model-epoch19.pt", map_location="cuda")

model, metric = train(dataset, model, batch_size=4, epochs=25, lr=0.01, mode=TrainingMode.DP, gc=1, vocab_size=vocab_size, save_model_on_epoch=True, checkpoint=checkpoint)